# CV 网格设计 PoC (A3.5)

对应计划 §A3.3 / §A3.5：在小样本上用 numpy 跑全量 **6 个候选标量**，可视化 16×16 热力图，回答一个关键决策问题：

> **FFT 高频能量占比**相较于 **Laplacian 方差** 是否提供额外信号？  
> 留 → B 阶段引入 FftSharp；砍 → 永久不实现。

输入：任意一批覆盖 §0.3 七类场景（拖影 / 点光源 / 细节丰富/平坦 / 纹理重复 / 噪声 等）的图片路径列表。

输出：每张图 × 每尺度 × 每标量 的 16×16 热力图；最终给 FFT 去留做人工判断。

---

**依赖**：`pip install numpy pillow matplotlib scipy`

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# --- 配置 ---
GRID = 16
PYRAMID_LEVELS = 3   # level 0 = 原图，level 1 = 1/2，level 2 = 1/4
FFT_CUTOFF_RATIO = 0.25  # 高频起始频率占奈奎斯特的比例

# TODO: 填 7 类场景各 1~2 张的路径
SAMPLES = [
    # Path(r'd:/path/to/sharp.jpg'),
    # Path(r'd:/path/to/motion_blur.jpg'),
    # Path(r'd:/path/to/point_light.jpg'),
]

In [ ]:
def load_luma(path: Path) -> np.ndarray:
    """按 Rec.709 载入单通道亮度，range [0, 255]。"""
    img = Image.open(path).convert('RGB')
    a = np.asarray(img, dtype=np.float32)
    return 0.2126 * a[..., 0] + 0.7152 * a[..., 1] + 0.0722 * a[..., 2]

def downsample_2x(luma: np.ndarray) -> np.ndarray:
    h, w = luma.shape
    h2, w2 = h // 2, w // 2
    return luma[:h2*2, :w2*2].reshape(h2, 2, w2, 2).mean(axis=(1, 3))

def cells_of(luma: np.ndarray):
    """按 16×16 切格，返回 generator of (gy, gx, cell_view)。边界用下取整。"""
    h, w = luma.shape
    ch, cw = h // GRID, w // GRID
    for gy in range(GRID):
        for gx in range(GRID):
            yield gy, gx, luma[gy*ch:(gy+1)*ch, gx*cw:(gx+1)*cw]

In [ ]:
from scipy.signal import convolve2d

LAP_K = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
SOBEL_Y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)

def scalar_laplacian_var(cell):
    resp = convolve2d(cell, LAP_K, mode='valid')
    return float(resp.var())

def scalar_sobel_mean(cell):
    gx = convolve2d(cell, SOBEL_X, mode='valid')
    gy = convolve2d(cell, SOBEL_Y, mode='valid')
    return float(np.sqrt(gx*gx + gy*gy).mean())

def scalar_grad_dir_entropy(cell, bins=8):
    gx = convolve2d(cell, SOBEL_X, mode='valid')
    gy = convolve2d(cell, SOBEL_Y, mode='valid')
    mag = np.sqrt(gx*gx + gy*gy)
    theta = np.arctan2(gy, gx)                   # [-π, π]
    norm = ((theta + np.pi) / np.pi) % 1.0       # 无向边对向合并
    hist, _ = np.histogram(norm, bins=bins, range=(0, 1), weights=mag)
    s = hist.sum()
    if s <= 0: return 0.0
    p = hist / s
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def scalar_luma_mean(cell):
    return float(cell.mean())

def scalar_luma_std(cell):
    return float(cell.std())

def scalar_fft_hf_ratio(cell, cutoff=FFT_CUTOFF_RATIO):
    """二期候选：高频能量占比（|F|² 在 cutoff × Nyquist 之外的占比）。"""
    f = np.fft.fftshift(np.fft.fft2(cell - cell.mean()))
    pw = (f.real**2 + f.imag**2)
    h, w = pw.shape
    cy, cx = h/2, w/2
    yy, xx = np.mgrid[0:h, 0:w]
    r = np.sqrt(((yy - cy) / cy) ** 2 + ((xx - cx) / cx) ** 2)
    total = pw.sum()
    if total <= 0: return 0.0
    return float(pw[r >= cutoff].sum() / total)

SCALARS = [
    ('laplacian_var', scalar_laplacian_var),
    ('sobel_mean', scalar_sobel_mean),
    ('grad_dir_entropy', scalar_grad_dir_entropy),
    ('luma_mean', scalar_luma_mean),
    ('luma_std', scalar_luma_std),
    ('fft_hf_ratio', scalar_fft_hf_ratio),     # 二期候选
]

In [ ]:
def compute_grid(luma: np.ndarray) -> dict[str, np.ndarray]:
    out = {name: np.zeros((GRID, GRID), dtype=np.float32) for name, _ in SCALARS}
    for gy, gx, cell in cells_of(luma):
        if cell.size < 9:
            continue
        for name, fn in SCALARS:
            out[name][gy, gx] = fn(cell)
    return out

def compute_pyramid(luma: np.ndarray):
    levels = [luma]
    for _ in range(PYRAMID_LEVELS - 1):
        levels.append(downsample_2x(levels[-1]))
    return [compute_grid(l) for l in levels]

In [ ]:
def show_sample(path: Path):
    luma = load_luma(path)
    grids = compute_pyramid(luma)

    fig, axes = plt.subplots(PYRAMID_LEVELS, len(SCALARS), figsize=(3*len(SCALARS), 3*PYRAMID_LEVELS))
    for p, grid in enumerate(grids):
        for s, (name, _) in enumerate(SCALARS):
            ax = axes[p, s]
            ax.imshow(grid[name], cmap='magma')
            ax.set_title(f'{name}\nlvl{p}')
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(path.name)
    fig.tight_layout()
    plt.show()

for p in SAMPLES:
    show_sample(Path(p))

## FFT 去留决策

人工对比每张图 `laplacian_var` 与 `fft_hf_ratio` 两列热力图：

- **FFT 热力图**对场景 7（点光源拖影、1-2px 级高频）展现出比 Laplacian 明显不同的模式 → 留，B 阶段引入 FftSharp
- **两列差异不显著**（相关系数 > 0.9 或视觉基本一致）→ 砍，二期也不做

记录结论到 `plans/dinov3-photo-ranking-plan.md` §A3.3 末尾。